In [1]:
from icalens import ICALens

lens = ICALens.from_pretrained("../local-icalens-models/official/icalens-gemma-2-2b-pile10k")

# Baseline

In [2]:
english_prompt = "Hangzhou is a historic city known for"
chinese_prompt = "杭州是一座历史悠久的城市，它以"

chinese_baseline = lens.generate(
    chinese_prompt,
    max_new_tokens=24,
)
english_baseline = lens.generate(
    english_prompt,
    max_new_tokens=24,
)
print("Chinese Baseline:", chinese_baseline)
print("English Baseline:", english_baseline)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Chinese Baseline: 其悠久的历史和丰富的文化遗产而闻名。杭州的文化遗产包括许多著名的景点，如西湖
English Baseline:  its beautiful scenery and rich culture. It is also a city with a long history of tea culture. The tea culture in


# Steering

In [3]:
steered = lens.generate(
    english_prompt,
    layer=20,
    initial_steer=(
        105,
        35.0,
        35.0,
    ),  # Steer the first generated token. Target: C105, add +35.0, but cap at +35.0
    steer=(105, 0.0, 30.0),  # Steer the rest tokens. Target: C105, add +0.0, but cap at +30.0
    steering_scope="current-position",
    max_new_tokens=24,
    # debug=True,
)
print("English prompt:", english_prompt)
print("Steered to Chinese:", steered)

English prompt: Hangzhou is a historic city known for
Steered to Chinese: 其历史悠久的文化和美丽的风景。它位于中国浙江省，距离上海仅有100公里。


In [4]:
ablated = lens.generate(
    chinese_prompt,
    layer=20,
    clamp=(105, 0.0),  # Steer all generated tokens by clamping the target C105 to +0.0.
    max_new_tokens=24,
    # debug=True,
)
print("Chinese prompt:", chinese_prompt)
print("Ablated:", ablated)

Chinese prompt: 杭州是一座历史悠久的城市，它以
Ablated:  its rich history and culture as well as its beautiful scenery. The city is home to many famous attractions, including the West


# Additional Analysis

In [ ]:
lens.analyze("杭州是一座历史悠久的城市", layer=20)

In [ ]:
lens.component_profile(layer=20, component=105)